In [1]:
!pip -q install geopandas openpyxl requests


In [2]:
import requests
import geopandas as gpd
import pandas as pd
import json
from io import BytesIO

ward_url = (
    "https://tgrac.telangana.gov.in/arcgis/rest/services/"
    "GHMCHealth_Folder/GHMC_Health_Facilities/MapServer/2/query"
)

params = {
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "geojson"
}

response = requests.get(ward_url, params=params)

print("Status:", response.status_code)
print("Size:", len(response.content) / 1024, "KB")

Status: 400
Size: 0.587890625 KB


In [7]:
import requests
import geopandas as gpd
import pandas as pd
from shapely.geometry import shape, Polygon

ward_url = (
    "https://tgrac.telangana.gov.in/arcgis/rest/services/"
    "GHMCHealth_Folder/GHMC_Health_Facilities/MapServer/2/query"
)

params = {
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "json"
}

response = requests.get(ward_url, params=params, timeout=60)

print("HTTP Status:", response.status_code)
print("Response size:", round(len(response.content) / 1024, 2), "KB")

data = response.json()

print("Keys:", data.keys())

HTTP Status: 200
Response size: 1727.78 KB
Keys: dict_keys(['displayFieldName', 'fieldAliases', 'geometryType', 'spatialReference', 'fields', 'features'])


In [5]:
# Check whether ArcGIS returned an error
if "error" in data:
    print("ArcGIS Error:")
    print(data["error"])
else:
    print("Features returned:", len(data["features"]))

Features returned: 155


In [8]:
features = data["features"]

records = []

for feature in features:
    attributes = feature["attributes"].copy()
    geometry_data = feature["geometry"]

    # Assuming 'esriGeometryPolygon' based on the API response 'geometryType'
    if geometry_data and "rings" in geometry_data:
        exterior_ring = geometry_data["rings"][0]
        interior_rings = geometry_data["rings"][1:] if len(geometry_data["rings"]) > 1 else None
        geometry = Polygon(exterior_ring, interior_rings)
    else:
        # Handle cases where geometry might be missing or not a polygon
        geometry = None

    attributes["geometry"] = geometry
    records.append(attributes)

wards = gpd.GeoDataFrame(
    records,
    geometry="geometry",
    crs="EPSG:4326"
)

print("Rows:", len(wards))
print("Columns:")
print(wards.columns.tolist())

Rows: 155
Columns:
['OBJECTID', 'AREA_IN_SQ', 'ward', 'CIRCLE', 'ZONE', 'Shape_Length', 'Shape_Area', 'geometry']


In [13]:
wards.head()

,OBJECTID,AREA_IN_SQ,ward,CIRCLE,ZONE,Shape_Length,Shape_Area,geometry
0,1,2599171.20578681,127-RANGAREDDY NAGAR,25-QUTUBULLAPUR,KUKATPALLY ZONE,0.072335,0.000220,"POLYGON ((78.45031 17.48797, 78.45029 17.4879,..."
1,2,3030525.32688414,119-OLD BOWENPALLY,24-KUKATPALLY,KUKATPALLY ZONE,0.091499,0.000257,"POLYGON ((78.47869 17.48565, 78.47943 17.48571..."
2,3,1080056.46203159,145-SEETHAPHALMANDI,29-SECUNDERABAD,SECUNDERABAD ZONE,0.043072,0.000092,"POLYGON ((78.52062 17.4268, 78.52074 17.42655,..."
3,4,826577.400553971,146-BOUDHA NAGAR,29-SECUNDERABAD,SECUNDERABAD ZONE,0.036197,0.000071,"POLYGON ((78.52018 17.4193, 78.51998 17.41854,..."
4,5,2095399.59603649,147-BANSILALPET,30-BEGUMPET,SECUNDERABAD ZONE,0.057413,0.000178,"POLYGON ((78.50605 17.43299, 78.50621 17.433, ..."


In [10]:
print("Number of unique wards:", wards["ward"].nunique())

print(
    wards[["ward", "CIRCLE", "ZONE", "AREA_IN_SQ"]]
    .sort_values("ward")
    .head(30)
    .to_string(index=False)
)

Number of unique wards: 155
                        ward                             CIRCLE                 ZONE       AREA_IN_SQ
                     1-KAPRA                            1-KAPRA       L.B.NAGAR ZONE 8944372.18269375
                    10-UPPAL                            2-UPPAL       L.B.NAGAR ZONE 8335381.76252596
            100-SANATH NAGAR                     17-KHAIRATABAD    KHAIRATHABAD ZONE 2527218.65970457
               101-ERRAGADDA                      19-YOUSUFGUDA SERILINGAMPALLY ZONE 1558899.24979181
          102-RAHAMATH NAGAR                      19-YOUSUFGUDA SERILINGAMPALLY ZONE 1029968.93694225
               103-BORABANDA                      19-YOUSUFGUDA SERILINGAMPALLY ZONE 911769.741616043
                104-KONDAPUR                 20-SERILINGAMPALLY SERILINGAMPALLY ZONE 14648928.9829252
              105-GACHIBOWLI                 20-SERILINGAMPALLY SERILINGAMPALLY ZONE 49739784.9150654
         106-SERILINGAMPALLY                 20-SERILI

In [11]:
wards[["ward", "CIRCLE", "ZONE", "AREA_IN_SQ"]]

,ward,CIRCLE,ZONE,AREA_IN_SQ
0,127-RANGAREDDY NAGAR,25-QUTUBULLAPUR,KUKATPALLY ZONE,2599171.20578681
1,119-OLD BOWENPALLY,24-KUKATPALLY,KUKATPALLY ZONE,3030525.32688414
2,145-SEETHAPHALMANDI,29-SECUNDERABAD,SECUNDERABAD ZONE,1080056.46203159
3,146-BOUDHA NAGAR,29-SECUNDERABAD,SECUNDERABAD ZONE,826577.400553971
4,147-BANSILALPET,30-BEGUMPET,SECUNDERABAD ZONE,2095399.59603649
...,...,...,...,...
150,4-MEERPET H.B.COLONY,1-KAPRA,L.B.NAGAR ZONE,3014512.24134588
151,1-KAPRA,1-KAPRA,L.B.NAGAR ZONE,8944372.18269375
152,94-SHAIKPET,18-JUBILEE HILLS,KHAIRATHABAD ZONE,4392202.39162178
153,64-DATTATHREYA NAGAR,14-GOSHAMAHAL,KHAIRATHABAD ZONE,685037.180962379


In [51]:
import pandas as pd

wards["ward_id"] = (
    wards["ward"]
    .str.extract(r"^(\d+)", expand=False)
    .apply(pd.to_numeric, errors='coerce')
    .astype('Int64')
)

print(wards[["ward", "ward_id"]].sort_values("ward_id").to_string(index=False))

                        ward  ward_id
                     1-KAPRA        1
          2-DR.A.S.RAO NAGAR        2
               3-CHERLAPALLY        3
        4-MEERPET H.B.COLONY        4
                  5-MALLAPUR        5
                  6-NACHARAM        6
             7-CHILUKA NAGAR        7
                 8-HABSIGUDA        8
               9-RAMANTHAPUR        9
                    10-UPPAL       10
                   11-NAGOLE       11
              12-MANSOORABAD       12
              13-HAYATHNAGAR       13
          14-B.N.REDDY NAGAR       14
          15-VANASTHALIPURAM       15
            16-HASTHINAPURAM       16
                17-CHAMPAPET       17
              18-LINGOJIGUDA       18
              19-SAROORNAGAR       19
                20-R.K.PURAM       20
                 21-KOTHAPET       21
           22-CHAITHANYAPURI       22
             23-GADDIANNARAM       23
                 24-SAIDABAD       24
             25-MOOSARAMBAGH       25
            

In [18]:
print("Minimum ward ID:", wards["ward_id"].min())
print("Maximum ward ID:", wards["ward_id"].max())
print("Unique ward IDs:", wards["ward_id"].nunique())

Minimum ward ID: 1
Maximum ward ID: 150
Unique ward IDs: 150


In [19]:
all_ids = set(range(1, 156))
gis_ids = set(wards["ward_id"])

missing_ids = sorted(all_ids - gis_ids)
extra_ids = sorted(gis_ids - all_ids)

print("Missing IDs:", missing_ids)
print("Extra IDs:", extra_ids)

Missing IDs: [151, 152, 153, 154, 155]
Extra IDs: [<NA>]


In [20]:
wards_projected = wards.to_crs("EPSG:32644")

wards["area_sq_km"] = (
    wards_projected.geometry.area / 1_000_000
)

print(
    wards[["ward_id", "ward", "area_sq_km"]]
    .sort_values("ward_id")
    .head(20)
)

     ward_id                  ward  area_sq_km
151        1               1-KAPRA    8.954194
149        2    2-DR.A.S.RAO NAGAR    4.661705
143        3         3-CHERLAPALLY   14.795384
150        4  4-MEERPET H.B.COLONY    3.014512
142        5            5-MALLAPUR    6.942442
125        6            6-NACHARAM    5.000633
76         7       7-CHILUKA NAGAR    2.937032
35         8           8-HABSIGUDA    5.262074
77         9         9-RAMANTHAPUR    2.971369
52        10              10-UPPAL    8.637914
51        11             11-NAGOLE   11.068491
49        12        12-MANSOORABAD   11.006985
55        13        13-HAYATHNAGAR   10.130161
48        14    14-B.N.REDDY NAGAR    8.693891
56        15    15-VANASTHALIPURAM    3.657312
45        16      16-HASTHINAPURAM   -4.888828
58        17          17-CHAMPAPET    2.272337
44        18        18-LINGOJIGUDA    2.705281
47        19        19-SAROORNAGAR    2.292709
46        20          20-R.K.PURAM    2.455294


In [21]:
print("Invalid geometries:", (~wards.geometry.is_valid).sum())
print("Empty geometries:", wards.geometry.is_empty.sum())
print("Null geometries:", wards.geometry.isna().sum())

Invalid geometries: 7
Empty geometries: 0
Null geometries: 0


In [22]:
print("CRS:", wards.crs)

CRS: EPSG:4326


In [23]:
wards_base = wards[
    [
        "ward_id",
        "ward",
        "CIRCLE",
        "ZONE",
        "area_sq_km",
        "geometry"
    ]
].copy()

wards_base = wards_base.rename(
    columns={
        "ward": "ward_name",
        "CIRCLE": "circle",
        "ZONE": "zone"
    }
)

wards_base.to_file(
    "/content/ghmc_wards_current.geojson",
    driver="GeoJSON"
)

wards_base.drop(columns="geometry").to_csv(
    "/content/ghmc_wards_current.csv",
    index=False
)

print("Saved successfully.")
print("Rows:", len(wards_base))

Saved successfully.
Rows: 155


In [24]:
problem_wards = wards[wards["ward_id"].isna()][
    ["OBJECTID", "ward", "CIRCLE", "ZONE", "AREA_IN_SQ"]
]

print(problem_wards.to_string(index=False))

 OBJECTID                        ward                     CIRCLE ZONE       AREA_IN_SQ
       17             CONTONMENT AREA            CANTONMENT AREA      104537.176161127
       54 GRAMPANCHAYAT PEERZADI GUDA GRAMPANCHAYAT PEERZADIGUDA      126321.478566996
       55  KALAVANCHA GRAM PANCHAYATH  KALAVANCHA GRAMPANCHAYATH      302009.674536216
       69                          OU                         OU      4280659.77597252
       80                 BANDLA GUDA                 BANDLAGUDA       1354341.8361091


In [25]:
print("Total records:", len(wards))
print("Valid ward IDs:", wards["ward_id"].notna().sum())
print("Missing ward IDs:", wards["ward_id"].isna().sum())

Total records: 155
Valid ward IDs: 150
Missing ward IDs: 5


In [26]:
invalid_wards = wards[~wards.geometry.is_valid].copy()

print(
    invalid_wards[
        ["OBJECTID", "ward", "CIRCLE", "ZONE", "AREA_IN_SQ"]
    ].to_string(index=False)
)

 OBJECTID               ward                             CIRCLE                 ZONE       AREA_IN_SQ
        9       136-NEREDMET                      28-MALKAJGIRI    SECUNDERABAD ZONE 7384988.64301425
       30 111-BHARATHI NAGAR 22-RAMACHANDRAPURAM & PATANCHERUVU SERILINGAMPALLY ZONE  1991645.7160305
       35        143-TARNAKA                    29-SECUNDERABAD    SECUNDERABAD ZONE 2417361.76334641
       46   16-HASTHINAPURAM                        4-LB. NAGAR       L.B.NAGAR ZONE  4946566.2053745
       58    26-OLD MALAKPET                         6-MALAKPET       CHARMINAR ZONE 2362340.05686644
       66   133-MACHABOLARUM                           27-ALWAL      KUKATPALLY ZONE 12533312.1906244
       89      33-MOGHALPURA                        9-CHARMINAR       CHARMINAR ZONE 845739.600069305


In [27]:
for idx, row in invalid_wards.iterrows():
    print(
        "\nOBJECTID:", row["OBJECTID"],
        "| Ward:", row["ward"],
        "| Reason:",
        row.geometry.is_valid
    )


OBJECTID: 9 | Ward: 136-NEREDMET | Reason: False

OBJECTID: 30 | Ward: 111-BHARATHI NAGAR | Reason: False

OBJECTID: 35 | Ward: 143-TARNAKA | Reason: False

OBJECTID: 46 | Ward: 16-HASTHINAPURAM | Reason: False

OBJECTID: 58 | Ward: 26-OLD MALAKPET | Reason: False

OBJECTID: 66 | Ward: 133-MACHABOLARUM | Reason: False

OBJECTID: 89 | Ward: 33-MOGHALPURA | Reason: False


In [28]:
from shapely.validation import explain_validity

invalid_wards["geometry_problem"] = invalid_wards.geometry.apply(
    explain_validity
)

print(
    invalid_wards[
        ["ward", "geometry_problem"]
    ].to_string(index=False)
)

              ward                                       geometry_problem
      136-NEREDMET     Hole lies outside shell[78.534988588 17.493482305]
111-BHARATHI NAGAR     Hole lies outside shell[78.301419627 17.506977433]
       143-TARNAKA Hole lies outside shell[78.5468502300001 17.443117257]
  16-HASTHINAPURAM Hole lies outside shell[78.551991549 17.3350725940001]
   26-OLD MALAKPET Hole lies outside shell[78.498062497 17.3754323920001]
  133-MACHABOLARUM Hole lies outside shell[78.5039704690001 17.533764938]
     33-MOGHALPURA      Hole lies outside shell[78.481604539 17.35700302]


In [29]:
ward16 = wards[wards["ward_id"] == 16].copy()

print(ward16[
    ["ward", "AREA_IN_SQ", "Shape_Area"]
].to_string(index=False))

print("Valid:", ward16.geometry.iloc[0].is_valid)
print("Problem:", explain_validity(ward16.geometry.iloc[0]))

            ward      AREA_IN_SQ  Shape_Area
16-HASTHINAPURAM 4946566.2053745     0.00042
Valid: False
Problem: Hole lies outside shell[78.551991549 17.3350725940001]


In [30]:
wards_clean = wards.copy()

# Keep only numbered GHMC wards
wards_clean = wards_clean[
    wards_clean["ward_id"].notna()
].copy()

# Convert ward ID to integer
wards_clean["ward_id"] = wards_clean["ward_id"].astype(int)

print("Records after removing non-ward areas:", len(wards_clean))
print("Unique ward IDs:", wards_clean["ward_id"].nunique())

Records after removing non-ward areas: 150
Unique ward IDs: 150


In [31]:
from shapely.validation import make_valid

wards_clean["geometry"] = (
    wards_clean["geometry"]
    .apply(make_valid)
)

In [32]:
print(
    "Invalid geometries:",
    (~wards_clean.geometry.is_valid).sum()
)

print(
    "Empty geometries:",
    wards_clean.geometry.is_empty.sum()
)

print(
    "Null geometries:",
    wards_clean.geometry.isna().sum()
)

Invalid geometries: 0
Empty geometries: 0
Null geometries: 0


In [33]:
wards_projected = wards_clean.to_crs("EPSG:32644")

wards_clean["area_sq_km"] = (
    wards_projected.geometry.area / 1_000_000
)

In [34]:
print("Negative/zero areas:",
      (wards_clean["area_sq_km"] <= 0).sum())

print("Missing areas:",
      wards_clean["area_sq_km"].isna().sum())

print(
    wards_clean[
        ["ward_id", "ward", "area_sq_km"]
    ]
    .sort_values("ward_id")
    .head(20)
    .to_string(index=False)
)

Negative/zero areas: 0
Missing areas: 0
 ward_id                 ward  area_sq_km
       1              1-KAPRA    8.954194
       2   2-DR.A.S.RAO NAGAR    4.661705
       3        3-CHERLAPALLY   14.795384
       4 4-MEERPET H.B.COLONY    3.014512
       5           5-MALLAPUR    6.942442
       6           6-NACHARAM    5.000633
       7      7-CHILUKA NAGAR    2.937032
       8          8-HABSIGUDA    5.262074
       9        9-RAMANTHAPUR    2.971369
      10             10-UPPAL    8.637914
      11            11-NAGOLE   11.068491
      12       12-MANSOORABAD   11.006985
      13       13-HAYATHNAGAR   10.130161
      14   14-B.N.REDDY NAGAR    8.693891
      15   15-VANASTHALIPURAM    3.657312
      16     16-HASTHINAPURAM    4.948204
      17         17-CHAMPAPET    2.272337
      18       18-LINGOJIGUDA    2.705281
      19       19-SAROORNAGAR    2.292709
      20         20-R.K.PURAM    2.455294


In [35]:
expected_ids = set(range(1, 151))
actual_ids = set(wards_clean["ward_id"])

print("Expected wards:", len(expected_ids))
print("Actual wards:", len(actual_ids))

print("Missing ward IDs:",
      sorted(expected_ids - actual_ids))

print("Unexpected ward IDs:",
      sorted(actual_ids - expected_ids))

Expected wards: 150
Actual wards: 150
Missing ward IDs: []
Unexpected ward IDs: []


In [36]:
print(
    wards_clean[
        ["ward_id", "ward", "CIRCLE", "ZONE", "area_sq_km"]
    ]
    .sort_values("ward_id")
    .tail(10)
    .to_string(index=False)
)

 ward_id                ward          CIRCLE              ZONE  area_sq_km
     141   141-GOUTHAM NAGAR   28-MALKAJGIRI SECUNDERABAD ZONE    2.047755
     142       142-ADDAGUTTA 29-SECUNDERABAD SECUNDERABAD ZONE    2.023314
     143         143-TARNAKA 29-SECUNDERABAD SECUNDERABAD ZONE    2.416870
     144       144-METTUGUDA 29-SECUNDERABAD SECUNDERABAD ZONE    3.349307
     145 145-SEETHAPHALMANDI 29-SECUNDERABAD SECUNDERABAD ZONE    1.081904
     146    146-BOUDHA NAGAR 29-SECUNDERABAD SECUNDERABAD ZONE    0.838298
     147     147-BANSILALPET     30-BEGUMPET SECUNDERABAD ZONE    2.095400
     148     148-RAMGOPALPET     30-BEGUMPET SECUNDERABAD ZONE    1.662034
     149        149-BEGUMPET     30-BEGUMPET SECUNDERABAD ZONE    5.684548
     150   150-MONDA  MARKET     30-BEGUMPET SECUNDERABAD ZONE    2.553843


In [37]:
wards_base = wards_clean[
    [
        "ward_id",
        "ward",
        "CIRCLE",
        "ZONE",
        "area_sq_km",
        "geometry"
    ]
].copy()

wards_base = wards_base.rename(
    columns={
        "ward": "ward_name",
        "CIRCLE": "circle",
        "ZONE": "zone"
    }
)

In [38]:
wards_base.to_file(
    "/content/ghmc_wards_150_clean.geojson",
    driver="GeoJSON"
)

wards_base.drop(
    columns="geometry"
).to_csv(
    "/content/ghmc_wards_150_clean.csv",
    index=False
)

print("Clean GHMC ward files created.")
print("Rows:", len(wards_base))

Clean GHMC ward files created.
Rows: 150


In [39]:
import pandas as pd

census_file = "/content/DDW_PCA2805_2011_MDDS with UI.xlsx"

xls = pd.ExcelFile(census_file)

print("Available sheets:")
print(xls.sheet_names)

Available sheets:
['EB-2805']


In [40]:
census_raw = pd.read_excel(
    census_file,
    sheet_name="EB-2805",
    header=None
)

print("Shape:", census_raw.shape)

print("\nFirst 15 rows:")
display(census_raw.head(15))

Shape: (192, 94)

First 15 rows:


,0,1,2,3,4,5,6,7,8,9,...,84,85,86,87,88,89,90,91,92,93
0,State,District,Subdistt,Town/Village,Ward,EB,Level,Name,TRU,No_HH,...,MARG_AL_0_3_F,MARG_HH_0_3_P,MARG_HH_0_3_M,MARG_HH_0_3_F,MARG_OT_0_3_P,MARG_OT_0_3_M,MARG_OT_0_3_F,NON_WORK_P,NON_WORK_M,NON_WORK_F
1,28,536,00000,000000,0000,000000,DISTRICT,Hyderabad,Total,849051,...,175,2264,1123,1141,19793,13818,5975,2530026,972990,1557036
2,28,536,00000,000000,0000,000000,DISTRICT,Hyderabad,Rural,0,...,0,0,0,0,0,0,0,0,0,0
3,28,536,00000,000000,0000,000000,DISTRICT,Hyderabad,Urban,849051,...,175,2264,1123,1141,19793,13818,5975,2530026,972990,1557036
4,28,536,04496,000000,0000,000000,SUB-DISTRICT,Shaikpet,Total,57917,...,18,123,56,67,979,576,403,148928,56970,91958
5,28,536,04496,000000,0000,000000,SUB-DISTRICT,Shaikpet,Rural,0,...,0,0,0,0,0,0,0,0,0,0
6,28,536,04496,000000,0000,000000,SUB-DISTRICT,Shaikpet,Urban,57917,...,18,123,56,67,979,576,403,148928,56970,91958
7,28,536,04496,802918,0000,000000,TOWN,GHMC (M Corp. + OG) (Part),Urban,57917,...,18,123,56,67,979,576,403,148928,56970,91958
8,28,536,04496,802918,0000,000000,TOWN,GHMC (M Corp.) (Part),Urban,57917,...,18,123,56,67,979,576,403,148928,56970,91958
9,28,536,04496,802918,0104,000000,WARD,GHMC (M Corp.) (Part) WARD NO.-0104,Urban,7774,...,1,9,3,6,59,31,28,18440,6762,11678


In [41]:
census = census_raw.copy()

census.columns = census.iloc[0]
census = census.iloc[1:].reset_index(drop=True)

print("Shape:", census.shape)
print(census.columns.tolist())

Shape: (191, 94)
['State', 'District', 'Subdistt', 'Town/Village', 'Ward', 'EB', 'Level', 'Name', 'TRU', 'No_HH', 'TOT_P', 'TOT_M', 'TOT_F', 'P_06', 'M_06', 'F_06', 'P_SC', 'M_SC', 'F_SC', 'P_ST', 'M_ST', 'F_ST', 'P_LIT', 'M_LIT', 'F_LIT', 'P_ILL', 'M_ILL', 'F_ILL', 'TOT_WORK_P', 'TOT_WORK_M', 'TOT_WORK_F', 'MAINWORK_P', 'MAINWORK_M', 'MAINWORK_F', 'MAIN_CL_P', 'MAIN_CL_M', 'MAIN_CL_F', 'MAIN_AL_P', 'MAIN_AL_M', 'MAIN_AL_F', 'MAIN_HH_P', 'MAIN_HH_M', 'MAIN_HH_F', 'MAIN_OT_P', 'MAIN_OT_M', 'MAIN_OT_F', 'MARGWORK_P', 'MARGWORK_M', 'MARGWORK_F', 'MARG_CL_P', 'MARG_CL_M', 'MARG_CL_F', 'MARG_AL_P', 'MARG_AL_M', 'MARG_AL_F', 'MARG_HH_P', 'MARG_HH_M', 'MARG_HH_F', 'MARG_OT_P', 'MARG_OT_M', 'MARG_OT_F', 'MARGWORK_3_6_P', 'MARGWORK_3_6_M', 'MARGWORK_3_6_F', 'MARG_CL_3_6_P', 'MARG_CL_3_6_M', 'MARG_CL_3_6_F', 'MARG_AL_3_6_P', 'MARG_AL_3_6_M', 'MARG_AL_3_6_F', 'MARG_HH_3_6_P', 'MARG_HH_3_6_M', 'MARG_HH_3_6_F', 'MARG_OT_3_6_P', 'MARG_OT_3_6_M', 'MARG_OT_3_6_F', 'MARGWORK_0_3_P', 'MARGWORK_0_3_M', '

In [42]:
census_wards = census[
    census["Level"].astype(str).str.upper().eq("WARD")
].copy()

print("Census ward records:", len(census_wards))

Census ward records: 108


In [43]:
display(
    census_wards[
        ["District", "Subdistt", "Town/Village", "Ward", "Level", "Name", "TRU", "No_HH"]
    ].head(30)
)

,District,Subdistt,Town/Village,Ward,Level,Name,TRU,No_HH
8,536,04496,802918,0104,WARD,GHMC (M Corp.) (Part) WARD NO.-0104,Urban,7774
9,536,04496,802918,0105,WARD,GHMC (M Corp.) (Part) WARD NO.-0105,Urban,15740
10,536,04496,802918,0107,WARD,GHMC (M Corp.) (Part) WARD NO.-0107,Urban,13866
11,536,04496,802918,0109,WARD,GHMC (M Corp.) (Part) WARD NO.-0109,Urban,12367
12,536,04496,802918,0110,WARD,GHMC (M Corp.) (Part) WARD NO.-0110,Urban,8170
18,536,04497,802918,0099,WARD,GHMC (M Corp.) (Part) WARD NO.-0099,Urban,5459
19,536,04497,802918,0100,WARD,GHMC (M Corp.) (Part) WARD NO.-0100,Urban,9363
25,536,04498,802918,0145,WARD,GHMC (M Corp.) (Part) WARD NO.-0145,Urban,8240
26,536,04498,802918,0146,WARD,GHMC (M Corp.) (Part) WARD NO.-0146,Urban,6909
27,536,04498,802918,0147,WARD,GHMC (M Corp.) (Part) WARD NO.-0147,Urban,9489


In [44]:
population_columns = [
    col for col in census.columns
    if "TOT_P" in str(col).upper()
    or "POP" in str(col).upper()
]

print(population_columns)

['TOT_P']


In [45]:
for i, col in enumerate(census.columns):
    print(i, repr(col))

0 'State'
1 'District'
2 'Subdistt'
3 'Town/Village'
4 'Ward'
5 'EB'
6 'Level'
7 'Name'
8 'TRU'
9 'No_HH'
10 'TOT_P'
11 'TOT_M'
12 'TOT_F'
13 'P_06'
14 'M_06'
15 'F_06'
16 'P_SC'
17 'M_SC'
18 'F_SC'
19 'P_ST'
20 'M_ST'
21 'F_ST'
22 'P_LIT'
23 'M_LIT'
24 'F_LIT'
25 'P_ILL'
26 'M_ILL'
27 'F_ILL'
28 'TOT_WORK_P'
29 'TOT_WORK_M'
30 'TOT_WORK_F'
31 'MAINWORK_P'
32 'MAINWORK_M'
33 'MAINWORK_F'
34 'MAIN_CL_P'
35 'MAIN_CL_M'
36 'MAIN_CL_F'
37 'MAIN_AL_P'
38 'MAIN_AL_M'
39 'MAIN_AL_F'
40 'MAIN_HH_P'
41 'MAIN_HH_M'
42 'MAIN_HH_F'
43 'MAIN_OT_P'
44 'MAIN_OT_M'
45 'MAIN_OT_F'
46 'MARGWORK_P'
47 'MARGWORK_M'
48 'MARGWORK_F'
49 'MARG_CL_P'
50 'MARG_CL_M'
51 'MARG_CL_F'
52 'MARG_AL_P'
53 'MARG_AL_M'
54 'MARG_AL_F'
55 'MARG_HH_P'
56 'MARG_HH_M'
57 'MARG_HH_F'
58 'MARG_OT_P'
59 'MARG_OT_M'
60 'MARG_OT_F'
61 'MARGWORK_3_6_P'
62 'MARGWORK_3_6_M'
63 'MARGWORK_3_6_F'
64 'MARG_CL_3_6_P'
65 'MARG_CL_3_6_M'
66 'MARG_CL_3_6_F'
67 'MARG_AL_3_6_P'
68 'MARG_AL_3_6_M'
69 'MARG_AL_3_6_F'
70 'MARG_HH_3_6_P'
71 'MARG

In [46]:
print(
    census_wards.groupby(
        ["District", "Subdistt", "Town/Village"]
    ).size().to_string()
)

District  Subdistt  Town/Village
536       04496     802918           5
          04497     802918           2
          04498     802918           4
          04499     802919           8
          04500     802918           5
          04501     573968           1
                    802918          10
          04502     802918           5
          04503     802918           3
          04504     802918           6
          04505     802918           7
          04506     802918          12
          04507     802918           4
          04508     802918          13
          04509     802918           8
          04510     802918           6
          04511     802918           9


In [47]:
print(
    census_wards["Name"]
    .astype(str)
    .head(50)
    .to_string(index=False)
)

  GHMC (M Corp.) (Part) WARD NO.-0104
  GHMC (M Corp.) (Part) WARD NO.-0105
  GHMC (M Corp.) (Part) WARD NO.-0107
  GHMC (M Corp.) (Part) WARD NO.-0109
  GHMC (M Corp.) (Part) WARD NO.-0110
  GHMC (M Corp.) (Part) WARD NO.-0099
  GHMC (M Corp.) (Part) WARD NO.-0100
  GHMC (M Corp.) (Part) WARD NO.-0145
  GHMC (M Corp.) (Part) WARD NO.-0146
  GHMC (M Corp.) (Part) WARD NO.-0147
  GHMC (M Corp.) (Part) WARD NO.-0148
      Secunderabad (CB) WARD NO.-0001
      Secunderabad (CB) WARD NO.-0002
      Secunderabad (CB) WARD NO.-0003
      Secunderabad (CB) WARD NO.-0004
      Secunderabad (CB) WARD NO.-0005
      Secunderabad (CB) WARD NO.-0006
      Secunderabad (CB) WARD NO.-0007
      Secunderabad (CB) WARD NO.-0008
  GHMC (M Corp.) (Part) WARD NO.-0140
  GHMC (M Corp.) (Part) WARD NO.-0141
  GHMC (M Corp.) (Part) WARD NO.-0142
  GHMC (M Corp.) (Part) WARD NO.-0149
  GHMC (M Corp.) (Part) WARD NO.-0150
  GHMC (M Corp.) (Part) WARD NO.-0088
  GHMC (M Corp.) (Part) WARD NO.-0089
  GHMC (M Co

In [48]:
census_ward_list = census_wards[
    [
        "Subdistt",
        "Town/Village",
        "Ward",
        "Name",
        "TOT_P",
        "TOT_M",
        "TOT_F",
        "No_HH"
    ]
].copy()

census_ward_list["population_2011"] = pd.to_numeric(
    census_ward_list["TOT_P"],
    errors="coerce"
)

census_ward_list["ward_code"] = (
    census_ward_list["Ward"]
    .astype(str)
    .str.strip()
    .str.zfill(4)
)

print("Total Census wards:", len(census_ward_list))
print("Missing population:", census_ward_list["population_2011"].isna().sum())

display(
    census_ward_list[
        [
            "ward_code",
            "Subdistt",
            "Name",
            "population_2011",
            "No_HH"
        ]
    ].to_string(index=False)
)

Total Census wards: 108
Missing population: 0


'ward_code Subdistt                                  Name  population_2011 No_HH\n     0104    04496   GHMC (M Corp.) (Part) WARD NO.-0104            30701  7774\n     0105    04496   GHMC (M Corp.) (Part) WARD NO.-0105            69177 15740\n     0107    04496   GHMC (M Corp.) (Part) WARD NO.-0107            60103 13866\n     0109    04496   GHMC (M Corp.) (Part) WARD NO.-0109            51014 12367\n     0110    04496   GHMC (M Corp.) (Part) WARD NO.-0110            39937  8170\n     0099    04497   GHMC (M Corp.) (Part) WARD NO.-0099            20958  5459\n     0100    04497   GHMC (M Corp.) (Part) WARD NO.-0100            38112  9363\n     0145    04498   GHMC (M Corp.) (Part) WARD NO.-0145            34659  8240\n     0146    04498   GHMC (M Corp.) (Part) WARD NO.-0146            29920  6909\n     0147    04498   GHMC (M Corp.) (Part) WARD NO.-0147            41336  9489\n     0148    04498   GHMC (M Corp.) (Part) WARD NO.-0148            29455  6923\n     0001    04499       Se

In [49]:
# Current GHMC ward IDs
current_ids = set(wards_base["ward_id"])

# Census numeric ward codes
census_ids = set(
    pd.to_numeric(
        census_ward_list["ward_code"],
        errors="coerce"
    ).dropna().astype(int)
)

print("Current GHMC wards:", len(current_ids))
print("Census ward records:", len(census_ids))

print("\nCensus ward numbers that also exist as current GHMC IDs:")
print(sorted(current_ids.intersection(census_ids)))

print("\nCensus ward numbers NOT represented in current GHMC IDs:")
print(sorted(census_ids - current_ids))

print("\nCurrent GHMC ward IDs with no same-number Census ward:")
print(sorted(current_ids - census_ids))

Current GHMC wards: 150
Census ward records: 107

Census ward numbers that also exist as current GHMC IDs:
[1, 2, 3, 4, 5, 6, 7, 8, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 102, 103, 104, 105, 106, 107, 108, 109, 110, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150]

Census ward numbers NOT represented in current GHMC IDs:
[]

Current GHMC ward IDs with no same-number Census ward:
[9, 10, 11, 12, 13, 14, 15, 16, 17, 57, 58, 59, 60, 101, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139]


In [52]:
import requests
import ssl

url = "https://www.censusindia.gov.in/nada/index.php/catalog/120/download/130/DH_2011_2805_PART_A_DCHB_HYDERABAD.pdf"

# Bypass SSL certificate verification for this specific request
response = requests.get(url, timeout=60, verify=False)

print("Status:", response.status_code)
print("Size MB:", round(len(response.content) / (1024 * 1024), 2))
print("Content type:", response.headers.get("Content-Type"))

/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.censusindia.gov.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status: 200
Size MB: 7.36
Content type: application/octet-stream


In [56]:
dchb_file = "/content/DH_2011_2805_PART_A_DCHB_HYDERABAD.pdf"

with open(dchb_file, "wb") as f:
    f.write(response.content)

print("Saved:", dchb_file)

Saved: /content/DH_2011_2805_PART_A_DCHB_HYDERABAD.pdf


In [54]:
import os

print("File exists:", os.path.exists(dchb_file))
print("File size MB:", round(os.path.getsize(dchb_file) / (1024 * 1024), 2))

File exists: True
File size MB: 7.36


In [62]:
!pip install PyPDF2

import PyPDF2

reader = PyPDF2.PdfReader(dchb_file)

print("Total pages:", len(reader.pages))

# Get missing ward IDs from previous analysis
# Assuming `current_ids` and `census_ids` are available from previous cells
missing_gis_ward_ids = sorted(list(current_ids - census_ids))

search_terms_specific_wards = []
for ward_id in missing_gis_ward_ids:
    search_terms_specific_wards.append(f"WARD NO.-{ward_id:04d}")

# Include other general terms if still desired, but focus on specifics first
search_terms = [
    "GHMC",
    "Hyderabad Municipal Corporation",
    "Ward",
] + search_terms_specific_wards

for term in search_terms:
    matches = []

    for page_num, page in enumerate(reader.pages):
        text = page.extract_text() or ""

        if term.lower() in text.lower():
            matches.append(page_num + 1)

    print(f"\n{term}:")
    print(matches[:30])

Total pages: 238

GHMC:
[25, 26, 62, 63, 69, 72, 75, 76, 79, 80, 81, 84, 87, 88, 89, 90, 91, 107, 118, 119, 120, 121, 122, 123, 124, 125, 126, 128, 129, 130]

Hyderabad Municipal Corporation:
[25, 107]

Ward:
[13, 19, 20, 32, 34, 53, 59, 61, 64, 67, 95, 99]

WARD NO.-0009:
[]

WARD NO.-0010:
[]

WARD NO.-0011:
[]

WARD NO.-0012:
[]

WARD NO.-0013:
[]

WARD NO.-0014:
[]

WARD NO.-0015:
[]

WARD NO.-0016:
[]

WARD NO.-0017:
[]

WARD NO.-0057:
[]

WARD NO.-0058:
[]

WARD NO.-0059:
[]

WARD NO.-0060:
[]

WARD NO.-0101:
[]

WARD NO.-0111:
[]

WARD NO.-0112:
[]

WARD NO.-0113:
[]

WARD NO.-0114:
[]

WARD NO.-0115:
[]

WARD NO.-0116:
[]

WARD NO.-0117:
[]

WARD NO.-0118:
[]

WARD NO.-0119:
[]

WARD NO.-0120:
[]

WARD NO.-0121:
[]

WARD NO.-0122:
[]

WARD NO.-0123:
[]

WARD NO.-0124:
[]

WARD NO.-0125:
[]

WARD NO.-0126:
[]

WARD NO.-0127:
[]

WARD NO.-0128:
[]

WARD NO.-0129:
[]

WARD NO.-0130:
[]

WARD NO.-0131:
[]

WARD NO.-0132:
[]

WARD NO.-0133:
[]

WARD NO.-0134:
[]

WARD NO.-0135:
[]



In [59]:
for page_num, page in enumerate(reader.pages):
    text = page.extract_text() or ""

    if "WARD NO.-0001" in text.upper():
        print("\n" + "=" * 80)
        print("PAGE:", page_num + 1)
        print("=" * 80)
        print(text[:8000])

In [71]:
display(
    ghmc_census[
        [
            "ward_code",
            "ward_id_numeric",
            "Subdistt",
            "Name",
            "population_2011",
            "No_HH"
        ]
    ]
    .sort_values("ward_id_numeric")
    .reset_index(drop=True)
)

,ward_code,ward_id_numeric,Subdistt,Name,population_2011,No_HH
0,0018,18,04511,GHMC (M Corp.) (Part) WARD NO.-0018,25242,5758
1,0019,19,04511,GHMC (M Corp.) (Part) WARD NO.-0019,45226,10382
2,0020,20,04511,GHMC (M Corp.) (Part) WARD NO.-0020,45484,10789
3,0021,21,04511,GHMC (M Corp.) (Part) WARD NO.-0021,56751,11159
4,0022,22,04509,GHMC (M Corp.) (Part) WARD NO.-0022,47220,9160
...,...,...,...,...,...,...
94,0146,146,04498,GHMC (M Corp.) (Part) WARD NO.-0146,29920,6909
95,0147,147,04498,GHMC (M Corp.) (Part) WARD NO.-0147,41336,9489
96,0148,148,04498,GHMC (M Corp.) (Part) WARD NO.-0148,29455,6923
97,0149,149,04500,GHMC (M Corp.) (Part) WARD NO.-0149,31948,8050


In [72]:
current_ward_ids = set(wards_base["ward_id"])

census_ghmc_ids = set(
    ghmc_census["ward_id_numeric"]
    .dropna()
    .astype(int)
)

print("Current GHMC wards:", len(current_ward_ids))
print("Census GHMC wards:", len(census_ghmc_ids))

print("\nCensus GHMC ward IDs:")
print(sorted(census_ghmc_ids))

print("\nCurrent GHMC wards missing from Census:")
print(sorted(current_ward_ids - census_ghmc_ids))

print("\nCensus GHMC IDs not in current GHMC:")
print(sorted(census_ghmc_ids - current_ward_ids))

Current GHMC wards: 150
Census GHMC wards: 99

Census GHMC ward IDs:
[18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 102, 103, 104, 105, 106, 107, 108, 109, 110, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150]

Current GHMC wards missing from Census:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 57, 58, 59, 60, 101, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139]

Census GHMC IDs not in current GHMC:
[]


In [74]:
import requests
import geopandas as gpd
from shapely.geometry import shape, Polygon

ghmc_old_url = (
    "https://tgrac.telangana.gov.in/arcgis/rest/services/"
    "MusiRiver/Musi_Basin_Web_Publication_vector/MapServer/50/query"
)

params = {
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "json"
}

response = requests.get(ghmc_old_url, params=params, timeout=120)
response.raise_for_status()

data = response.json()

print("Features returned:", len(data.get("features", [])))

records = []

for feature in data["features"]:
    attrs = feature["attributes"].copy()
    geometry_data = feature["geometry"]

    # Assuming 'esriGeometryPolygon' based on the API response 'geometryType'
    if geometry_data and "rings" in geometry_data:
        exterior_ring = geometry_data["rings"][0]
        interior_rings = geometry_data["rings"][1:] if len(geometry_data["rings"]) > 1 else None
        geometry = Polygon(exterior_ring, interior_rings)
    else:
        # Handle cases where geometry might be missing or not a polygon
        geometry = None

    attrs["geometry"] = geometry
    records.append(attrs)

ghmc_wards_tgrac = gpd.GeoDataFrame(
    records,
    geometry="geometry",
    crs="EPSG:4326"
)

print("Rows:", len(ghmc_wards_tgrac))
print("Columns:")
print(ghmc_wards_tgrac.columns.tolist())

Features returned: 155
Rows: 155
Columns:
['FID', 'AREA_IN_SQ', 'ward', 'CIRCLE', 'ZONE', 'test', 'geometry']


In [75]:
display(
    ghmc_wards_tgrac[
        ["ward", "CIRCLE", "ZONE", "geometry"]
    ].head(20)
)

,ward,CIRCLE,ZONE,geometry
0,127-RANGAREDDY NAGAR,25-QUTUBULLAPUR,KUKATPALLY ZONE,"POLYGON ((78.45031 17.48797, 78.45029 17.4879,..."
1,119-OLD BOWENPALLY,24-KUKATPALLY,KUKATPALLY ZONE,"POLYGON ((78.47869 17.48565, 78.47943 17.48571..."
2,145-SEETHAPHALMANDI,29-SECUNDERABAD,SECUNDERABAD ZONE,"POLYGON ((78.52062 17.4268, 78.52074 17.42655,..."
3,146-BOUDHA NAGAR,29-SECUNDERABAD,SECUNDERABAD ZONE,"POLYGON ((78.52018 17.4193, 78.51998 17.41854,..."
4,147-BANSILALPET,30-BEGUMPET,SECUNDERABAD ZONE,"POLYGON ((78.50605 17.43299, 78.50621 17.433, ..."
5,144-METTUGUDA,29-SECUNDERABAD,SECUNDERABAD ZONE,"POLYGON ((78.53594 17.4344, 78.53611 17.43423,..."
6,148-RAMGOPALPET,30-BEGUMPET,SECUNDERABAD ZONE,"POLYGON ((78.48803 17.44018, 78.48848 17.43902..."
7,150-MONDA MARKET,30-BEGUMPET,SECUNDERABAD ZONE,"POLYGON ((78.50581 17.45198, 78.50633 17.45176..."
8,136-NEREDMET,28-MALKAJGIRI,SECUNDERABAD ZONE,"POLYGON ((78.53099 17.48953, 78.53169 17.4901,..."
9,141-GOUTHAM NAGAR,28-MALKAJGIRI,SECUNDERABAD ZONE,"POLYGON ((78.53626 17.44931, 78.53632 17.44917..."


In [76]:
print("Number of records:", len(ghmc_wards_tgrac))
print("Unique ward names:", ghmc_wards_tgrac["ward"].nunique())

print("\nSample ward values:")
print(
    ghmc_wards_tgrac["ward"]
    .dropna()
    .astype(str)
    .head(30)
    .tolist()
)

Number of records: 155
Unique ward names: 155

Sample ward values:
['127-RANGAREDDY NAGAR', '119-OLD BOWENPALLY', '145-SEETHAPHALMANDI', '146-BOUDHA NAGAR', '147-BANSILALPET', '144-METTUGUDA', '148-RAMGOPALPET', '150-MONDA  MARKET', '136-NEREDMET', '141-GOUTHAM NAGAR', '140-MALKAJGIRI', '139-EAST ANAND BAGH', '142-ADDAGUTTA', '137-VINAYAK NAGAR', '134-ALWAL', '132-JEEDIMETLA', 'CONTONMENT AREA', '120-BALA NAGAR', '130-SUBHASH NAGAR', '131-QUTHBULLAPUR', '121-KUKATPALLY', '125-GAJULARAMARAM', '124-ALLYWN COLONY', '114-K.P.H.B.COLONY', '116-ALLAPUR', '115-BALAJI NAGAR', '117-MOOSAPET', '105-GACHIBOWLI', '110-CHANDANAGAR', '111-BHARATHI NAGAR']


In [77]:
rangareddy_file = "/content/DDW_PCA2806_2011_MDDS with UI.xlsx"

In [78]:
import pandas as pd

xls_rr = pd.ExcelFile(rangareddy_file)

print(xls_rr.sheet_names)

['EB-2806']


In [79]:
rangareddy_raw = pd.read_excel(
    rangareddy_file,
    sheet_name="EB-2806",
    header=None
)

print("Shape:", rangareddy_raw.shape)

display(rangareddy_raw.head(12))

Shape: (1168, 94)


,0,1,2,3,4,5,6,7,8,9,...,84,85,86,87,88,89,90,91,92,93
0,State,District,Subdistt,Town/Village,Ward,EB,Level,Name,TRU,No_HH,...,MARG_AL_0_3_F,MARG_HH_0_3_P,MARG_HH_0_3_M,MARG_HH_0_3_F,MARG_OT_0_3_P,MARG_OT_0_3_M,MARG_OT_0_3_F,NON_WORK_P,NON_WORK_M,NON_WORK_F
1,28,537,00000,000000,0000,000000,DISTRICT,Rangareddy,Total,1235341,...,6847,4194,2090,2104,21148,12692,8456,3100663,1212092,1888571
2,28,537,00000,000000,0000,000000,DISTRICT,Rangareddy,Rural,349140,...,6122,1019,341,678,4166,2187,1979,798230,358025,440205
3,28,537,00000,000000,0000,000000,DISTRICT,Rangareddy,Urban,886201,...,725,3175,1749,1426,16982,10505,6477,2302433,854067,1448366
4,28,537,04512,000000,0000,000000,SUB-DISTRICT,Marpalle,Total,10961,...,182,19,10,9,51,23,28,24326,11385,12941
5,28,537,04512,000000,0000,000000,SUB-DISTRICT,Marpalle,Rural,10961,...,182,19,10,9,51,23,28,24326,11385,12941
6,28,537,04512,000000,0000,000000,SUB-DISTRICT,Marpalle,Urban,0,...,0,0,0,0,0,0,0,0,0,0
7,28,537,04512,573969,0000,000000,VILLAGE,Jamshedapur,Rural,154,...,0,0,0,0,0,0,0,309,140,169
8,28,537,04512,573970,0000,000000,VILLAGE,Ghanpur,Rural,263,...,10,1,1,0,2,1,1,548,250,298
9,28,537,04512,573971,0000,000000,VILLAGE,Mogiligundla,Rural,219,...,6,0,0,0,1,0,1,438,218,220


In [81]:
# Prepare Rangareddy Census data
census_rr = rangareddy_raw.copy()

# First row contains the actual column names
census_rr.columns = census_rr.iloc[0]
census_rr = census_rr.iloc[1:].reset_index(drop=True)

print("Shape:", census_rr.shape)

# Identify ward-level records
census_rr_wards = census_rr[
    census_rr["Level"]
    .astype(str)
    .str.upper()
    .eq("WARD")
].copy()

print("Rangareddy Census ward records:", len(census_rr_wards))

Shape: (1167, 94)
Rangareddy Census ward records: 137


In [82]:
display(
    census_rr_wards[
        [
            "District",
            "Subdistt",
            "Town/Village",
            "Ward",
            "Level",
            "Name",
            "TRU",
            "No_HH",
            "TOT_P"
        ]
    ]
    .head(50)
)

,District,Subdistt,Town/Village,Ward,Level,Name,TRU,No_HH,TOT_P
119,537,04516,802918,0111,WARD,GHMC (M Corp.) (Part) WARD NO.-0111,Urban,11784,54122
120,537,04516,802918,0112,WARD,GHMC (M Corp.) (Part) WARD NO.-0112,Urban,22527,93141
121,537,04516,802918,0113,WARD,GHMC (M Corp.) (Part) WARD NO.-0113,Urban,19378,78284
122,537,04516,802918,0114,WARD,GHMC (M Corp.) (Part) WARD NO.-0114,Urban,20925,83773
128,537,04517,802918,0101,WARD,GHMC (M Corp.) (Part) WARD NO.-0101,Urban,6733,28195
129,537,04517,802918,0117,WARD,GHMC (M Corp.) (Part) WARD NO.-0117,Urban,18596,69083
130,537,04517,802918,0118,WARD,GHMC (M Corp.) (Part) WARD NO.-0118,Urban,12392,52236
131,537,04517,802918,0119,WARD,GHMC (M Corp.) (Part) WARD NO.-0119,Urban,20206,83665
132,537,04517,802918,0120,WARD,GHMC (M Corp.) (Part) WARD NO.-0120,Urban,9586,41373
133,537,04517,802918,0121,WARD,GHMC (M Corp.) (Part) WARD NO.-0121,Urban,16514,71076


In [83]:
ghmc_rr = census_rr_wards[
    census_rr_wards["Name"]
    .astype(str)
    .str.upper()
    .str.contains("GHMC", na=False)
].copy()

ghmc_rr["ward_code"] = (
    ghmc_rr["Ward"]
    .astype(str)
    .str.strip()
    .str.zfill(4)
)

ghmc_rr["ward_id_numeric"] = pd.to_numeric(
    ghmc_rr["ward_code"],
    errors="coerce"
)

ghmc_rr["population_2011"] = pd.to_numeric(
    ghmc_rr["TOT_P"],
    errors="coerce"
)

print("Rangareddy GHMC records:", len(ghmc_rr))
print("Unique ward codes:", ghmc_rr["ward_code"].nunique())
print("Missing population:", ghmc_rr["population_2011"].isna().sum())

Rangareddy GHMC records: 54
Unique ward codes: 50
Missing population: 0


In [84]:
display(
    ghmc_rr[
        [
            "ward_code",
            "ward_id_numeric",
            "Subdistt",
            "Name",
            "population_2011",
            "No_HH"
        ]
    ]
    .sort_values("ward_id_numeric")
    .reset_index(drop=True)
)

,ward_code,ward_id_numeric,Subdistt,Name,population_2011,No_HH
0,0001,1,04522,GHMC (M Corp.) WARD NO.-0001,44149,10899
1,0002,2,04522,GHMC (M Corp.) WARD NO.-0002,45264,11222
2,0002,2,04523,GHMC (M Corp.) WARD NO.-0002,13900,3288
3,0003,3,04524,GHMC (M Corp.) (Part) WARD NO.-0003,57747,14042
4,0004,4,04524,GHMC (M Corp.) (Part) WARD NO.-0004,67130,16611
5,0005,5,04524,GHMC (M Corp.) (Part) WARD NO.-0005,57688,14013
6,0006,6,04524,GHMC (M Corp.) (Part) WARD NO.-0006,52167,12750
7,0007,7,04524,GHMC (M Corp.) (Part) WARD NO.-0007,59061,14425
8,0008,8,04524,GHMC (M Corp.) (Part) WARD NO.-0008,75227,18366
9,0009,9,04524,GHMC (M Corp.) (Part) WARD NO.-0009,10744,2527


In [85]:
# Search the complete Rangareddy Census dataset for wards 0115 and 0116

missing_rr = census_rr[
    census_rr["Ward"]
    .astype(str)
    .str.strip()
    .isin(["0115", "0116"])
].copy()

display(
    missing_rr[
        [
            "District",
            "Subdistt",
            "Town/Village",
            "Ward",
            "Level",
            "Name",
            "TRU",
            "No_HH",
            "TOT_P"
        ]
    ]
)

,District,Subdistt,Town/Village,Ward,Level,Name,TRU,No_HH,TOT_P


In [86]:
missing_hyd = census[
    census["Ward"]
    .astype(str)
    .str.strip()
    .isin(["0115", "0116"])
].copy()

display(
    missing_hyd[
        [
            "District",
            "Subdistt",
            "Town/Village",
            "Ward",
            "Level",
            "Name",
            "TRU",
            "No_HH",
            "TOT_P"
        ]
    ]
)

,District,Subdistt,Town/Village,Ward,Level,Name,TRU,No_HH,TOT_P


In [87]:
# Search Census records for any mention of ward 115 or 116
for label, df in [
    ("Hyderabad", census),
    ("Rangareddy", census_rr)
]:
    print(f"\n===== {label} =====")

    mask = (
        df.astype(str)
        .apply(
            lambda col: col.str.contains(
                r"0115|0116|WARD NO\.-?115|WARD NO\.-?116",
                case=False,
                regex=True,
                na=False
            )
        )
        .any(axis=1)
    )

    result = df[mask].copy()

    if result.empty:
        print("No matching records found.")
    else:
        display(
            result[
                [
                    "District",
                    "Subdistt",
                    "Town/Village",
                    "Ward",
                    "Level",
                    "Name",
                    "TRU",
                    "No_HH",
                    "TOT_P"
                ]
            ]
        )


===== Hyderabad =====
No matching records found.

===== Rangareddy =====
No matching records found.


In [88]:
# ============================================
# Build 2011 GHMC ward-level population table
# ============================================

# Hyderabad Census GHMC wards
hyd_population = (
    ghmc_census[
        ["ward_id_numeric", "population_2011", "No_HH"]
    ]
    .copy()
    .rename(columns={
        "ward_id_numeric": "ward_id",
        "No_HH": "households_2011"
    })
)

# Rangareddy Census GHMC wards
# Some wards are split across Census subdistricts,
# so aggregate them to one ward-level record.
rr_population = (
    ghmc_rr[
        ["ward_id_numeric", "population_2011", "No_HH"]
    ]
    .groupby("ward_id_numeric", as_index=False)
    .agg({
        "population_2011": "sum",
        "No_HH": "sum"
    })
    .rename(columns={
        "ward_id_numeric": "ward_id",
        "No_HH": "households_2011"
    })
)

# Combine Hyderabad + Rangareddy
census_population = pd.concat(
    [hyd_population, rr_population],
    ignore_index=True
)

# Check for duplicate ward IDs
duplicates = (
    census_population[
        census_population["ward_id"].duplicated(keep=False)
    ]
    .sort_values("ward_id")
)

print("Hyderabad records:", len(hyd_population))
print("Rangareddy aggregated records:", len(rr_population))
print("Combined records:", len(census_population))
print("Unique ward IDs:", census_population["ward_id"].nunique())

print("\nDuplicate ward IDs after combining:")
display(duplicates)

Hyderabad records: 99
Rangareddy aggregated records: 50
Combined records: 149
Unique ward IDs: 148

Duplicate ward IDs after combining:


,ward_id,population_2011,households_2011
10,148,29455,6923
148,148,21871,5460


In [89]:
# ============================================
# Final 2011 Census population by GHMC ward
# ============================================

census_population = (
    pd.concat(
        [hyd_population, rr_population],
        ignore_index=True
    )
    .groupby("ward_id", as_index=False)
    .agg({
        "population_2011": "sum",
        "households_2011": "sum"
    })
)

# Make sure ward IDs are integers
census_population["ward_id"] = (
    census_population["ward_id"].astype(int)
)

# Sort
census_population = census_population.sort_values(
    "ward_id"
).reset_index(drop=True)

print("Final Census ward records:", len(census_population))
print("Unique ward IDs:", census_population["ward_id"].nunique())

print("\nMissing current GHMC wards:")
missing_wards = sorted(
    set(range(1, 151)) -
    set(census_population["ward_id"])
)
print(missing_wards)

print("\nDuplicate ward IDs:")
print(
    census_population[
        census_population["ward_id"].duplicated(keep=False)
    ]
)

print("\nWard 148:")
display(
    census_population[
        census_population["ward_id"] == 148
    ]
)

Final Census ward records: 148
Unique ward IDs: 148

Missing current GHMC wards:
[115, 116]

Duplicate ward IDs:
Empty DataFrame
Columns: [ward_id, population_2011, households_2011]
Index: []

Ward 148:


,ward_id,population_2011,households_2011
145,148,51326,12383


In [90]:
# ============================================
# Create GHMC 150-Ward Master Dataset
# ============================================

master = wards_base.merge(
    census_population,
    on="ward_id",
    how="left",
    validate="one_to_one"
)

# Calculate 2011 population density
master["population_density_2011"] = (
    master["population_2011"] /
    master["area_sq_km"]
)

# Sort by ward
master = master.sort_values(
    "ward_id"
).reset_index(drop=True)

# ============================================
# Validation
# ============================================

print("Rows:", len(master))
print("Unique ward IDs:", master["ward_id"].nunique())

print("\nMissing population:")
print(master["population_2011"].isna().sum())

print("\nMissing households:")
print(master["households_2011"].isna().sum())

print("\nMissing population density:")
print(master["population_density_2011"].isna().sum())

print("\nMissing population wards:")
display(
    master.loc[
        master["population_2011"].isna(),
        ["ward_id", "ward_name", "circle", "zone", "area_sq_km"]
    ]
)

print("\nMaster dataset preview:")
display(
    master[
        [
            "ward_id",
            "ward_name",
            "circle",
            "zone",
            "area_sq_km",
            "population_2011",
            "households_2011",
            "population_density_2011"
        ]
    ].head(10)
)

Rows: 150
Unique ward IDs: 150

Missing population:
2

Missing households:
2

Missing population density:
2

Missing population wards:


,ward_id,ward_name,circle,zone,area_sq_km
114,115,115-BALAJI NAGAR,23-MOOSAPET,KUKATPALLY ZONE,8.755052
115,116,116-ALLAPUR,23-MOOSAPET,KUKATPALLY ZONE,3.273570



Master dataset preview:


,ward_id,ward_name,circle,zone,area_sq_km,population_2011,households_2011,population_density_2011
0,1,1-KAPRA,1-KAPRA,L.B.NAGAR ZONE,8.954194,44149.0,10899,4930.538598
1,2,2-DR.A.S.RAO NAGAR,1-KAPRA,L.B.NAGAR ZONE,4.661705,59164.0,14510,12691.492926
2,3,3-CHERLAPALLY,1-KAPRA,L.B.NAGAR ZONE,14.795384,57747.0,14042,3903.041738
3,4,4-MEERPET H.B.COLONY,1-KAPRA,L.B.NAGAR ZONE,3.014512,67130.0,16611,22268.943040
4,5,5-MALLAPUR,1-KAPRA,L.B.NAGAR ZONE,6.942442,57688.0,14013,8309.468584
5,6,6-NACHARAM,1-KAPRA,L.B.NAGAR ZONE,5.000633,52167.0,12750,10432.079885
6,7,7-CHILUKA NAGAR,2-UPPAL,L.B.NAGAR ZONE,2.937032,59061.0,14425,20109.078126
7,8,8-HABSIGUDA,2-UPPAL,L.B.NAGAR ZONE,5.262074,75227.0,18366,14296.074936
8,9,9-RAMANTHAPUR,2-UPPAL,L.B.NAGAR ZONE,2.971369,68161.0,16118,22939.260894
9,10,10-UPPAL,2-UPPAL,L.B.NAGAR ZONE,8.637914,68115.0,16304,7885.583973


In [91]:
# ============================================
# Save GHMC 150-Ward Master Dataset
# ============================================

# GeoJSON
master.to_file(
    "/content/hyderabad_ward_master_2011.geojson",
    driver="GeoJSON"
)

# CSV
master.drop(columns="geometry").to_csv(
    "/content/hyderabad_ward_master_2011.csv",
    index=False
)

print("Master files created successfully.")
print()
print("GeoJSON:")
print("/content/hyderabad_ward_master_2011.geojson")

print()
print("CSV:")
print("/content/hyderabad_ward_master_2011.csv")

print()
print("Rows:", len(master))
print("Columns:", len(master.columns))

Master files created successfully.

GeoJSON:
/content/hyderabad_ward_master_2011.geojson

CSV:
/content/hyderabad_ward_master_2011.csv

Rows: 150
Columns: 9


In [92]:
# ============================================
# Final Master Dataset Quality Check
# ============================================

print("===== MASTER DATASET QUALITY CHECK =====")

print("Rows:", master.shape[0])
print("Columns:", master.shape[1])

print("\nWard IDs:")
print("Min:", master["ward_id"].min())
print("Max:", master["ward_id"].max())
print("Unique:", master["ward_id"].nunique())

print("\nGeometry:")
print("Null:", master.geometry.isna().sum())
print("Empty:", master.geometry.is_empty.sum())
print("Invalid:", (~master.geometry.is_valid).sum())

print("\nPopulation:")
print("Available:", master["population_2011"].notna().sum())
print("Missing:", master["population_2011"].isna().sum())

print("\nPopulation density:")
print("Available:", master["population_density_2011"].notna().sum())
print("Missing:", master["population_density_2011"].isna().sum())

print("\nMissing population wards:")
print(
    master.loc[
        master["population_2011"].isna(),
        "ward_id"
    ].tolist()
)

===== MASTER DATASET QUALITY CHECK =====
Rows: 150
Columns: 9

Ward IDs:
Min: 1
Max: 150
Unique: 150

Geometry:
Null: 0
Empty: 0
Invalid: 0

Population:
Available: 148
Missing: 2

Population density:
Available: 148
Missing: 2

Missing population wards:
[115, 116]


In [93]:
from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Create project folder
drive_folder = "/content/drive/MyDrive/Hyderabad_Urban_Heat_Analytics/data/processed"
os.makedirs(drive_folder, exist_ok=True)

# Copy the finalized master files
shutil.copy(
    "/content/hyderabad_ward_master_2011.geojson",
    drive_folder
)

shutil.copy(
    "/content/hyderabad_ward_master_2011.csv",
    drive_folder
)

print("Files saved successfully!")
print("\nSaved location:")
print(drive_folder)

print("\nFiles:")
print(os.listdir(drive_folder))

Mounted at /content/drive
Files saved successfully!

Saved location:
/content/drive/MyDrive/Hyderabad_Urban_Heat_Analytics/data/processed

Files:
['hyderabad_ward_master_2011.geojson', 'hyderabad_ward_master_2011.csv']
